# Study 905 — Residual Reversal — the teardown

The residual construction, the residual-vs-raw race, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_weeks': 808, 'fp': '357fd262912f', 'spread_bps': -0.38, 't_nw': -0.05, 't_1s': -0.04, 'lo_bps': 34.12, 'hi_bps': 34.51, 'welch_t': -0.03, 'gross_sharpe': -0.01, 'hit': 51.1, 'raw_bps': -0.03, 'raw_t': -0.0, 'noscreen_bps': 2.31, 'noscreen_t': 0.32, 'placebo_obs': -0.38, 'placebo_mean': 0.063, 'placebo_sd': 5.348, 'placebo_p': 0.513, 'placebo_draws': 1000, 'era_early_bps': 2.18, 'era_early_t': 0.26, 'era_early_n': 364, 'era_late_bps': -3.18, 'era_late_t': -0.23, 'era_late_n': 443, 'timer_1_gross': -0.38, 'timer_1_cost': 2.96, 'timer_1_net': -3.35, 'timer_1_t': -0.39, 'timer_5_gross': -0.38, 'timer_5_cost': 10.96, 'timer_5_net': -11.35, 'timer_5_t': -1.32, 'null_mean_t': 0.17, 'null_sd_t': 0.78, 'null_fire': 0, 'planted_t': 24.99, 'planted_raw_t': 13.23}

## The headline — long residual-loser / short residual-winner, liquid subset

Weekly equal-weight bottom-30% minus top-30% market-model-residual spread, top-60% by trailing dollar volume. The RAW weekly reversal is the foil.

In [2]:
print(f"residual reversal : {R['spread_bps']:+.2f} bps/wk  NW(8) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}  (n={R['n_weeks']} wks)")
print(f"  books           : loser {R['lo_bps']:+.2f} vs winner {R['hi_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f}), gross Sharpe {R['gross_sharpe']:.2f}, hit {R['hit']:.1f}%")
print(f"raw reversal (foil): {R['raw_bps']:+.2f} bps/wk  NW t = {R['raw_t']:+.2f}")
print(f"residual, NO screen: {R['noscreen_bps']:+.2f} bps/wk  NW t = {R['noscreen_t']:+.2f}")

residual reversal : -0.38 bps/wk  NW(8) t = -0.05  one-sample t = -0.04  (n=808 wks)
  books           : loser +34.12 vs winner +34.51 bps (Welch t = -0.03), gross Sharpe -0.01, hit 51.1%
raw reversal (foil): -0.03 bps/wk  NW t = -0.00
residual, NO screen: +2.31 bps/wk  NW t = +0.32


## Placebo — column-permute the forward returns (1,000 permutations)

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> p = {R['placebo_p']:.3f}  (dead centre)")

observed -0.38 bps vs placebo mean +0.063 (sd 5.348) -> p = 0.513  (dead centre)


## Robustness — two eras (split 2018-01-01)

In [4]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")
print('  -> the sign flips across eras; nothing to stand on.')

2010-2017 (n=364): +2.18 bps  NW t = +0.26
2018-2026 (n=443): -3.18 bps  NW t = -0.23
  -> the sign flips across eras; nothing to stand on.


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per weekly rebalance; short pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/wk (cost {c:.2f}/wk, t={t:+.2f})")

 1 bp one-way: gross -0.38 -> net -3.35 bps/wk (cost 2.96/wk, t=-0.39)
5 bps one-way: gross -0.38 -> net -11.35 bps/wk (cost 10.96/wk, t=-1.32)


## Synthetic positive control — the cleaner is real, the null is real

Live: the residual detector must recover a planted residual reversal, beat the factor-muddied raw detector, and NOT fire on the null.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from resid_reversal import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=905+s, n_assets=40, n_days=1600))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: residual NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
plan = st.synthetic_detect(data.synthetic_panel(edge=0.35, seed=905, n_assets=40, n_days=2000))
print(f"planted (edge=0.35): residual NW t = {plan['t_nw']:+.2f}  >>  raw NW t = {plan['raw_t_nw']:+.2f} (muddied)")

null (edge=0), 8 seeds: residual NW t mean +0.38 (sd 0.67), |t|>=2 in 0/8
planted (edge=0.35): residual NW t = +24.99  >>  raw NW t = +13.23 (muddied)


## Verdict

- **Signal — None.** On 50 liquid US mega-caps the factor-cleaned, liquidity-screened weekly residual reversal earns **-0.38 bps/week** (NW *t* = **-0.05**) — a flat line. The raw foil is equally dead (-0.03 bps), the placebo is dead-centre (*p* = 0.51), and the two eras flip sign (+0.26 / -0.23). The 20-seed synthetic control recovers a *planted* residual reversal cleanly (*t* = 25, above the factor-muddied raw *t* = 13; fires on 0/20 nulls), so the null is real, not machinery: short-term reversal lives in small/illiquid breadth, not in mega-caps. *Survivorship biases the magnitude upward — and it is still zero.*
- **Tradability — Mirage.** A gross edge of essentially zero cannot survive the 2.96 bps/week round-trip friction of a fully-turning weekly book: net **-3.35 bps/wk** at 1 bp one-way, **-11.35** at 5 bps. Even the slightly-positive un-screened gross (+2.31 bps) is eaten many times over.